# Visual Learning metadata tables

Builds the two metadata tables the problem sets use to choose a session, in the same
pattern as `V1DD_metadata.ipynb` and `bci_metadata.ipynb`.

| output | one row per |
| --- | --- |
| `visual_learning_session_metadata.csv` | session |
| `visual_learning_plane_metadata.csv` | session x imaging plane |

Both are written to `/code/metadata/`. Run this notebook when a new processing batch
lands; the problem sets read the CSVs, not docDB.


In [1]:
import os
import time
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

OUTPUT_DIR = '/code/metadata'
DATA_DIR = '/data'
CAPSULE_MOUNT = 'Visual-Learning-SWDB' 

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)


https://api.allenneuraldynamics.org/v1/metadata_index/data_assets


In [3]:
# The cohort is defined by subject, not by project_name: five different
# project_name values are interleaved across the same mice.
CTL_MICE = ['782149', '790322', '788406', '800792', '800995', '804363']

# Processed asset names end in _processed_<date>_<time>. Anchor at end-of-string --
# `_processed_` also appears mid-name in further-derived assets (coreg, czstack).
PROCESSED_PATTERN = (r'^multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+'
                     r'_processed_\d{4}-\d{2}-\d{2}_[\d-]+$')


# Function to aggregate metadata
def agg(pipeline, tries=5, base_sleep=8):
    """Run a docDB aggregation, retrying the gateway's intermittent 503s."""
    last = None
    for attempt in range(tries):
        try:
            return docdb_api_client.aggregate_docdb_records(pipeline=pipeline)
        except Exception as exc:
            last = exc
            if attempt < tries - 1:
                time.sleep(base_sleep * (attempt + 1))
    raise RuntimeError(f'docDB aggregate failed after {tries} attempts: {last}')

## Get metadata 

docDB holds both aind-data-schema generations in the same collection, and they put the
session-level fields in different places. There is no queryable version field &mdash; the
generation is whichever top-level block is a populated object:

| | v1 | v2 |
| --- | --- | --- |
| session block | `session` | `acquisition` |
| session type | `session.session_type` | `acquisition.acquisition_type` |
| start time | `session.session_start_time` | `acquisition.acquisition_start_time` |
| per-plane streams | `session.data_streams` | `acquisition.data_streams` |
| genotype | `subject.genotype` | `subject.subject_details.genotype` |

**Querying the wrong generation fails silently.** Every projected field comes back `None`,
the columns vanish from the DataFrame rather than raising, and you get an empty table that
looks like a successful run. So detect first rather than assuming &mdash; and re-run this
cell after any migration instead of trusting a hardcoded choice.


In [4]:
# Count assets by which block is populated. No $type in the $match stage -- see below.
probe = agg([
    {'$match': {'data_description.subject_id': {'$in': CTL_MICE},
                'name': {'$regex': '^multiplane-ophys_'}}},
    {'$project': {'shape': {'$cond': [
        {'$eq': [{'$type': '$session'}, 'object']}, 'v1_session',
        {'$cond': [{'$eq': [{'$type': '$acquisition'}, 'object']},
                   'v2_acquisition', 'neither']}]}}},
    {'$group': {'_id': '$shape', 'n': {'$sum': 1}}},
])
counts = {row['_id']: row['n'] for row in probe}
print('ophys assets by schema shape:', counts)

SCHEMA = 'v2' if counts.get('v2_acquisition', 0) > counts.get('v1_session', 0) else 'v1'
print('using', SCHEMA, 'field paths')

# One place to change when the migration lands.
FIELDS = {
    'v1': {'block': 'session',
           'type_field': 'session.session_type',
           'start': 'session.session_start_time',
           'end': 'session.session_end_time',
           'rig': 'session.rig_id',
           'streams': 'session.data_streams',
           'genotype': 'subject.genotype',
           'sex': 'subject.sex',
           'dob': 'subject.date_of_birth'},
    'v2': {'block': 'acquisition',
           'type_field': 'acquisition.acquisition_type',
           'start': 'acquisition.acquisition_start_time',
           'end': 'acquisition.acquisition_end_time',
           'rig': 'acquisition.instrument_id',
           'streams': 'acquisition.data_streams',
           'genotype': 'subject.subject_details.genotype',
           'sex': 'subject.subject_details.sex',
           'dob': 'subject.subject_details.date_of_birth'},
}[SCHEMA]


ophys assets by schema shape: {'v1_session': 2053, 'neither': 36}
using v1 field paths


## Session table

Filter on plain indexed fields here and do the filename filtering in pandas afterwards.


In [5]:
def session_pipeline(subject_id):
    return [
        {'$match': {'data_description.subject_id': subject_id,
                    'data_description.data_level': 'derived',
                    FIELDS['type_field']: {'$exists': True}}},
        {'$project': {
            'name': 1,
            'subject_id': '$data_description.subject_id',
            'project_name': '$data_description.project_name',
            'session_type': f"${FIELDS['type_field']}",
            'session_start_time': f"${FIELDS['start']}",
            'session_end_time': f"${FIELDS['end']}",
            'rig': f"${FIELDS['rig']}",
            'genotype': f"${FIELDS['genotype']}",
            'sex': f"${FIELDS['sex']}",
            'date_of_birth': f"${FIELDS['dob']}",
        }},
    ]


records = []
for mouse in CTL_MICE:
    rows = agg(session_pipeline(mouse))
    records.extend(rows)
    print(f'{mouse}: {len(rows)} processed assets')

sessions = pd.DataFrame(records)
print(f'\n{len(sessions)} rows before deduplication')

missing = [c for c in ('session_type', 'session_start_time', 'genotype')
           if c not in sessions.columns]
if missing:
    raise RuntimeError(
        f'{missing} absent from every row -- the {SCHEMA} field paths did not match '
        f'this data. Re-run the schema probe above.')


782149: 402 processed assets
790322: 331 processed assets
788406: 494 processed assets
800792: 245 processed assets
800995: 231 processed assets
804363: 193 processed assets

1896 rows before deduplication


### One row per session, newest processing only

A session is reprocessed whenever the pipeline changes, so the same session appears
several times under different `_processed_` stamps. Keep the newest.


In [6]:
sessions = sessions[sessions.name.str.match(PROCESSED_PATTERN)].copy()

sessions['session_id'] = sessions.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
sessions['processed_stamp'] = sessions.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

sessions = (sessions.sort_values('processed_stamp')
                    .drop_duplicates('session_id', keep='last'))

print(f'{len(sessions)} unique sessions across {sessions.subject_id.nunique()} mice')
print(sessions.subject_id.value_counts().sort_index().to_string())


158 unique sessions across 6 mice
subject_id
782149    24
788406    32
790322    24
800792    29
800995    27
804363    22


In [7]:
# Dates and ages, as in the other metadata notebooks
sessions['session_date'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).date())
sessions['session_time'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).time())
sessions['date_of_birth'] = sessions.date_of_birth.map(
    lambda x: datetime.strptime(x, '%Y-%m-%d').date() if isinstance(x, str) else x)
sessions['age_days'] = [(d - b).days if pd.notnull(b) else np.nan
                        for d, b in zip(sessions.session_date, sessions.date_of_birth)]

# Training stage and image set, parsed from session_type
sessions['stage'] = sessions.session_type.str.extract(
    r'^(TRAINING_\d|OPHYS_\d|STAGE_\d)')
sessions['image_set'] = sessions.session_type.str.extract(r'_images_([AB])')
sessions['session_number'] = (sessions.sort_values('session_date')
                                      .groupby('subject_id').cumcount() + 1)

order = ['subject_id', 'session_id', 'name', 'session_type', 'stage', 'image_set',
         'session_number', 'session_date', 'session_time', 'age_days',
         'genotype', 'sex', 'date_of_birth', 'rig', 'project_name',
         'processed_stamp', '_id']
sessions = (sessions[[c for c in order if c in sessions.columns]]
            .sort_values(['subject_id', 'session_date'])
            .reset_index(drop=True))

sessions.head(10)


,subject_id,session_id,name,session_type,stage,image_set,session_number,session_date,session_time,age_days,genotype,sex,date_of_birth,rig,project_name,processed_stamp,_id
0,782149,multiplane-ophys_782149_2025-03-25_09-46-08,multiplane-ophys_782149_2025-03-25_09-46-08_pr...,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,09:46:08.591468,108,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.2,LearningmFISHTask1A,2026-08-19_00-32-51,24865b94-ffd8-4eb9-b062-70fa46a17b39
1,782149,multiplane-ophys_782149_2025-03-28_10-55-25,multiplane-ophys_782149_2025-03-28_10-55-25_pr...,TRAINING_1_gratings,TRAINING_1,NaN,2,2025-03-28,10:55:25.569080,111,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-34-09,2bef1d11-dc35-4b8b-b825-7145bb0fe603
2,782149,multiplane-ophys_782149_2025-03-29_10-10-29,multiplane-ophys_782149_2025-03-29_10-10-29_pr...,TRAINING_1_gratings,TRAINING_1,NaN,3,2025-03-29,10:10:29.493070,112,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,Learning mFISH-V1omFISH,2026-08-19_00-33-54,c1a6d250-dea5-4de5-8185-29a92e405878
3,782149,multiplane-ophys_782149_2025-03-31_12-23-33,multiplane-ophys_782149_2025-03-31_12-23-33_pr...,TRAINING_1_gratings,TRAINING_1,NaN,4,2025-03-31,12:23:33.753970,114,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-34-28,edbe415f-da52-4f07-86b3-957fb399c4a6
4,782149,multiplane-ophys_782149_2025-04-01_09-42-11,multiplane-ophys_782149_2025-04-01_09-42-11_pr...,TRAINING_2_gratings_flashed,TRAINING_2,NaN,5,2025-04-01,09:42:11.814685,115,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-33-58,b670a19a-435f-401b-a63c-53ce8bd9c29f
5,782149,multiplane-ophys_782149_2025-04-02_12-16-24,multiplane-ophys_782149_2025-04-02_12-16-24_pr...,TRAINING_3_images_A_10uL_reward,TRAINING_3,A,6,2025-04-02,12:16:24.347832,116,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-34-02,81e1e0cc-2326-405b-b508-cf3b564341e2
6,782149,multiplane-ophys_782149_2025-04-04_11-39-31,multiplane-ophys_782149_2025-04-04_11-39-31_pr...,TRAINING_3_images_A_10uL_reward,TRAINING_3,A,7,2025-04-04,11:39:31.012983,118,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,Learning mFISH-V1omFISH,2026-08-19_00-34-04,0c7751aa-8f88-44b0-b41d-9549729a73b6
7,782149,multiplane-ophys_782149_2025-04-08_09-28-39,multiplane-ophys_782149_2025-04-08_09-28-39_pr...,TRAINING_3_images_A_10uL_reward,TRAINING_3,A,8,2025-04-08,09:28:39.721364,122,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-34-01,c4f77a4a-2212-4c31-bdfe-ed52fb96c1cd
8,782149,multiplane-ophys_782149_2025-04-09_09-46-45,multiplane-ophys_782149_2025-04-09_09-46-45_pr...,TRAINING_4_images_A_training,TRAINING_4,A,9,2025-04-09,09:46:45.121922,123,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-33-57,050cad7b-523b-4343-bcd8-e0a9a2de4fd2
9,782149,multiplane-ophys_782149_2025-04-10_09-26-55,multiplane-ophys_782149_2025-04-10_09-26-55_pr...,TRAINING_5_images_A_epilogue,TRAINING_5,A,10,2025-04-10,09:26:55.042787,124,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,2026-08-19_00-34-32,66bacb53-530a-4932-8cda-1639e9c81422


## Plane metadata

QC and imaging geometry are **per plane**, not per session, so we need to grab this info. It comes from `$unwind`-ing
`session.data_streams` and then `ophys_fovs`.



In [8]:
def plane_pipeline(subject_id):
    streams = FIELDS['streams']
    return [
        {'$match': {'data_description.subject_id': subject_id,
                    'data_description.data_level': 'derived',
                    FIELDS['type_field']: {'$exists': True}}},
        {'$unwind': f'${streams}'},
        {'$unwind': f'${streams}.ophys_fovs'},
        {'$project': {
            'name': 1,
            'subject_id': '$data_description.subject_id',
            'session_type': f"${FIELDS['type_field']}",
            'fov_index': f'${streams}.ophys_fovs.index',
            'targeted_structure': f'${streams}.ophys_fovs.targeted_structure',
            'imaging_depth': f'${streams}.ophys_fovs.imaging_depth',
            'frame_rate': f'${streams}.ophys_fovs.frame_rate',
            'scanimage_roi_index': f'${streams}.ophys_fovs.scanimage_roi_index',
        }},
    ]


plane_records, failed = [], []
for mouse in CTL_MICE:
    try:
        rows = agg(plane_pipeline(mouse))
        plane_records.extend(rows)
        print(f'{mouse}: {len(rows)} plane rows')
    except Exception as exc:
        failed.append(mouse)
        print(f'{mouse}: FAILED -- {str(exc)[:80]}')

if failed:
    print(f'\nRETRY THESE: {failed}  (gateway 503s, not a data problem)')

planes = pd.DataFrame(plane_records)
print(f'\n{len(planes)} plane rows')


782149: 3216 plane rows
790322: 2648 plane rows
788406: 3952 plane rows
800792: 1774 plane rows
800995: 1680 plane rows
804363: 1544 plane rows

14814 plane rows


In [9]:
# Same dedup as the session table, then join the session columns on
planes = planes[planes.name.str.match(PROCESSED_PATTERN)].copy()
planes['session_id'] = planes.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
planes['processed_stamp'] = planes.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

# keep only the newest processing generation per session, matching `sessions`
keep = set(zip(sessions.session_id, sessions.processed_stamp))
planes = planes[[(s, p) in keep for s, p in
                 zip(planes.session_id, planes.processed_stamp)]]

planes['plane_name'] = (planes.targeted_structure.astype(str) + '_'
                       + planes.fov_index.astype(str))

planes = planes.merge(
    sessions[['session_id', 'subject_id', 'session_date', 'stage', 'image_set',
              'session_number', 'genotype']],
    on=['session_id', 'subject_id'], how='left', suffixes=('', '_session'))

order = ['subject_id', 'session_id', 'session_type', 'stage', 'image_set',
         'session_number', 'session_date', 'plane_name', 'fov_index',
         'targeted_structure', 'imaging_depth', 'frame_rate',
         'scanimage_roi_index', 'genotype', 'name']
planes = (planes[[c for c in order if c in planes.columns]]
          .sort_values(['subject_id', 'session_date', 'fov_index'])
          .reset_index(drop=True))
print(f'{len(planes)} planes across {planes.session_id.nunique()} sessions')
planes.head(10)


1216 planes across 158 sessions


,subject_id,session_id,session_type,stage,image_set,session_number,session_date,plane_name,fov_index,targeted_structure,imaging_depth,frame_rate,scanimage_roi_index,genotype,name
0,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_0,0,VISp,40,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
1,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_1,1,VISp,320,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
2,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_2,2,VISp,80,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
3,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_3,3,VISp,280,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
4,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_4,4,VISp,120,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
5,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_5,5,VISp,240,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
6,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_6,6,VISp,160,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
7,782149,multiplane-ophys_782149_2025-03-25_09-46-08,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,VISp_7,7,VISp,200,10.63,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-25_09-46-08_pr...
8,782149,multiplane-ophys_782149_2025-03-28_10-55-25,TRAINING_1_gratings,TRAINING_1,NaN,2,2025-03-28,VISp_0,0,VISp,160,9.48,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-28_10-55-25_pr...
9,782149,multiplane-ophys_782149_2025-03-28_10-55-25,TRAINING_1_gratings,TRAINING_1,NaN,2,2025-03-28,VISp_1,1,VISp,200,9.48,0,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,multiplane-ophys_782149_2025-03-28_10-55-25_pr...


## Restrict to what is actually attached

docDB knows about every processed asset; the capsule only mounts some of them. The
problem sets can only open a file that is on `/data`, so filter both tables to the
assets present in the mount &mdash; the same step `bci_metadata.ipynb` does with
`os.listdir`.

If the mount is not attached this cell says so and leaves the tables unfiltered,
rather than silently emitting empty CSVs.


In [10]:
mount_path = os.path.join(DATA_DIR, CAPSULE_MOUNT)

if os.path.isdir(mount_path):
    attached = set(os.listdir(mount_path))
    print(f'{len(attached)} entries in {CAPSULE_MOUNT}')

    # The mount may be keyed by asset name or by session id -- accept either.
    in_mount = (sessions.name.isin(attached)
                | sessions.session_id.isin(attached)
                | sessions.session_id.map(
                    lambda s: any(a.startswith(s) for a in attached)))
    print(f'{int(in_mount.sum())} of {len(sessions)} sessions present in the mount')

    if in_mount.any():
        sessions['in_capsule'] = in_mount
        planes['in_capsule'] = planes.session_id.isin(
            sessions.loc[in_mount, 'session_id'])
    else:
        print('WARNING: no session names matched the mount contents.')
        print('Sample mount entries:', sorted(attached)[:3])
        sessions['in_capsule'] = False
        planes['in_capsule'] = False
else:
    print(f'{mount_path} not attached to this capsule -- tables not filtered.')
    sessions['in_capsule'] = np.nan
    planes['in_capsule'] = np.nan


155 entries in Visual-Learning-SWDB
155 of 158 sessions present in the mount


In [11]:
print(len(sessions))
sessions = sessions[sessions.in_capsule == True]
print(len(sessions))

158
155


In [12]:
print(len(planes))
planes = planes[planes.in_capsule == True]
print(len(planes))

1216
1192


In [13]:
# Drop test sessions with only 2 planes 

n_planes = pd.DataFrame(planes.groupby('name').count()['session_id'])
sessions_to_drop = n_planes[n_planes.session_id!=8].index.unique()

print(len(sessions))
sessions = sessions[sessions.name.isin(sessions_to_drop)==False]
print(len(sessions))

print(len(planes))
planes = planes[planes.name.isin(sessions_to_drop)==False]
print(len(planes))

155
147
1192
1176


## Z-drift QC

Each processed asset carries a `quality_control` block with per-plane `Z-drift Analysis`
evaluations. Two things to know about its shape before trusting a query against it:

- The metrics are nested one level deeper than the evaluation, and **each evaluation holds
  metrics for every plane**, not just the one named in the evaluation title. So the plane a
  metric refers to comes from `metrics.name`, not from `evaluations.name`.
- `metrics.name` is not written consistently across processing generations: some rows read
  `VISp_4 Z-drift Analysis`, others `VISp_4 Z-drift Analysis - VISp_4`. Parse the leading
  plane name rather than splitting on the suffix.

Status lives in `status_history`, which is a list &mdash; take the last entry, since a metric
can be re-evaluated (Automated first, then a human).

In [14]:
def zdrift_pipeline(subject_id):
    """Per-plane z-drift metrics for one mouse's processed assets."""
    return [
        {'$match': {'data_description.subject_id': subject_id,
                    'data_description.data_level': 'derived',
                    'name': {'$regex': r'^multiplane-ophys_.*_processed_'}}},
        # evaluations -> the individual z-drift evaluations
        {'$unwind': '$quality_control.evaluations'},
        {'$match': {'quality_control.evaluations.name': {'$regex': '^Z-drift'}}},
        # metrics -> one row per PLANE (each evaluation carries all planes)
        {'$unwind': '$quality_control.evaluations.metrics'},
        {'$project': {
            'name': 1,
            'metric_name': '$quality_control.evaluations.metrics.name',
            'z_drift_um': '$quality_control.evaluations.metrics.value.z_drift_um',
            'status_history': '$quality_control.evaluations.metrics.status_history',
        }},
    ]


zdrift_records, no_qc = [], []
for mouse in CTL_MICE:
    rows = agg(zdrift_pipeline(mouse))
    if not rows:
        no_qc.append(mouse)
    zdrift_records.extend(rows)
    print(f'{mouse}: {len(rows)} z-drift metric rows')

if no_qc:
    print(f'\nNO z-drift QC AT ALL for: {no_qc}')
    print('Their sessions will show z_drift_qc_status = "no QC" below -- absence of a '
          'failure is not evidence of passing.')

zdrift = pd.DataFrame(zdrift_records)
print(f'\n{len(zdrift)} metric rows total')

782149: 384 z-drift metric rows
790322: 392 z-drift metric rows
788406: 552 z-drift metric rows
800792: 262 z-drift metric rows
800995: 360 z-drift metric rows
804363: 184 z-drift metric rows

2134 metric rows total


In [15]:
# Plane name = the leading token of metric_name. Handles both spellings:
#   'VISp_4 Z-drift Analysis'  and  'VISp_4 Z-drift Analysis - VISp_4'
zdrift['plane_name'] = zdrift.metric_name.str.extract(r'^(\w+_\d+)')

# status_history is a list; the LAST entry is the current verdict.
zdrift['status'] = zdrift.status_history.map(
    lambda h: h[-1]['status'] if isinstance(h, list) and h else None)

# Same dedup as the other tables: newest processing generation only.
zdrift['session_id'] = zdrift.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
zdrift['processed_stamp'] = zdrift.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')
keep = set(zip(sessions.session_id, sessions.processed_stamp))
zdrift = zdrift[[(s, p) in keep for s, p in
                 zip(zdrift.session_id, zdrift.processed_stamp)]]

# One asset can repeat a plane across several evaluations -- collapse to one row.
zdrift = zdrift.drop_duplicates(['session_id', 'plane_name'])

print('planes parsed:', sorted(zdrift.plane_name.dropna().unique()))
print('status counts:', zdrift.status.value_counts().to_dict())
print(f'{len(zdrift)} plane-level rows across {zdrift.session_id.nunique()} sessions')
zdrift[['session_id', 'plane_name', 'z_drift_um', 'status']].head(10)

planes parsed: ['VISp_0', 'VISp_1', 'VISp_2', 'VISp_3', 'VISp_4', 'VISp_5', 'VISp_6', 'VISp_7']
status counts: {'Pass': 729, 'Fail': 111}
840 plane-level rows across 105 sessions


,session_id,plane_name,z_drift_um,status
192,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_0,9.00,Pass
193,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_1,5.25,Pass
194,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_2,6.75,Pass
195,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_3,2.25,Pass
196,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_4,6.75,Pass
197,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_5,3.75,Pass
198,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_6,9.75,Pass
199,multiplane-ophys_782149_2025-04-02_12-16-24,VISp_7,4.50,Pass
200,multiplane-ophys_782149_2025-05-02_10-55-09,VISp_0,6.00,Pass
201,multiplane-ophys_782149_2025-05-02_10-55-09,VISp_1,8.25,Pass


### Fold the per-plane detail into the session table

The session table gets one row per session, so the plane-level facts are collapsed into
list-valued columns. Three of them come from the plane table built above (`plane_names`,
`imaging_depths`, `targeted_structures`) and one from the z-drift QC
(`planes_failing_zdrift`).

`z_drift_qc_status` distinguishes three cases that are easy to conflate: every plane passed,
some plane failed, or **no QC exists for this session at all**. The third is not a pass.

In [16]:
# --- per-session plane summaries, from the plane table ---
plane_summary = (planes.sort_values('fov_index')
                       .groupby('session_id')
                       .agg(plane_names=('plane_name', list),
                            imaging_depths=('imaging_depth', list),
                            targeted_structures=('targeted_structure',
                                                 lambda s: sorted(set(s.dropna()))),
                            n_planes=('plane_name', 'size'))
                       .reset_index())
print('plane summary rows:', len(plane_summary))

# --- per-session z-drift failures ---
failed = (zdrift[zdrift.status == 'Fail']
          .sort_values('plane_name')
          .groupby('session_id')['plane_name'].apply(list)
          .rename('planes_failing_zdrift').reset_index())

# Sessions that HAVE z-drift QC at all -- needed to tell "passed" from "never checked".
has_qc = set(zdrift.session_id.unique())
print(f'{len(failed)} sessions have at least one plane failing z-drift; '
      f'{len(has_qc)} sessions have z-drift QC at all')

# --- merge onto the session table ---
sessions = sessions.merge(plane_summary, on='session_id', how='left')
sessions = sessions.merge(failed, on='session_id', how='left')

# Empty list rather than NaN, so the column has one type throughout.
sessions['planes_failing_zdrift'] = sessions.planes_failing_zdrift.map(
    lambda v: v if isinstance(v, list) else [])

sessions['n_planes_failing_zdrift'] = sessions.planes_failing_zdrift.map(len)
sessions['z_drift_qc_status'] = [
    'no QC' if sid not in has_qc else ('fail' if n else 'pass')
    for sid, n in zip(sessions.session_id, sessions.n_planes_failing_zdrift)]

print('\nz_drift_qc_status:', sessions.z_drift_qc_status.value_counts().to_dict())
print('sessions with >=1 failing plane:', int((sessions.n_planes_failing_zdrift > 0).sum()))

plane summary rows: 147
40 sessions have at least one plane failing z-drift; 105 sessions have z-drift QC at all

z_drift_qc_status: {'pass': 65, 'no QC': 42, 'fail': 40}
sessions with >=1 failing plane: 40


In [17]:
# `acquisition_date` is `session_date` under a clearer name -- the date the data
# was collected, not the date it was processed (that is `processed_stamp`).
sessions['acquisition_date'] = sessions.session_date

order = ['subject_id', 'session_id', 'name', 'session_type', 'stage', 'image_set',
         'session_number', 'acquisition_date', 'session_date', 'session_time',
         'age_days', 'genotype', 'sex', 'date_of_birth', 'rig', 'project_name',
         'n_planes', 'plane_names', 'imaging_depths', 'targeted_structures',
         'z_drift_qc_status', 'n_planes_failing_zdrift', 'planes_failing_zdrift',
         'processed_stamp', '_id']
sessions = sessions[[c for c in order if c in sessions.columns]]

print(sessions.columns.tolist())
sessions[['session_id', 'acquisition_date', 'n_planes', 'plane_names',
          'imaging_depths', 'targeted_structures', 'z_drift_qc_status',
          'planes_failing_zdrift']].head(8)

['subject_id', 'session_id', 'name', 'session_type', 'stage', 'image_set', 'session_number', 'acquisition_date', 'session_date', 'session_time', 'age_days', 'genotype', 'sex', 'date_of_birth', 'rig', 'project_name', 'n_planes', 'plane_names', 'imaging_depths', 'targeted_structures', 'z_drift_qc_status', 'n_planes_failing_zdrift', 'planes_failing_zdrift', 'processed_stamp', '_id']


,session_id,acquisition_date,n_planes,plane_names,imaging_depths,targeted_structures,z_drift_qc_status,planes_failing_zdrift
0,multiplane-ophys_782149_2025-03-25_09-46-08,2025-03-25,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[40, 320, 80, 280, 120, 240, 160, 200]",[VISp],pass,[]
1,multiplane-ophys_782149_2025-03-28_10-55-25,2025-03-28,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 114, 244, 80, 280, 40, 310]",[VISp],pass,[]
2,multiplane-ophys_782149_2025-03-29_10-10-29,2025-03-29,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[158, 198, 114, 246, 80, 276, 43, 306]",[VISp],fail,"[VISp_0, VISp_4, VISp_6]"
3,multiplane-ophys_782149_2025-03-31_12-23-33,2025-03-31,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 245, 80, 280, 40, 310]",[VISp],pass,[]
4,multiplane-ophys_782149_2025-04-01_09-42-11,2025-04-01,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 240, 85, 280, 40, 300]",[VISp],fail,"[VISp_4, VISp_6]"
5,multiplane-ophys_782149_2025-04-02_12-16-24,2025-04-02,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 116, 245, 80, 280, 40, 310]",[VISp],pass,[]
6,multiplane-ophys_782149_2025-04-04_11-39-31,2025-04-04,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 240, 80, 280, 40, 305]",[VISp],fail,"[VISp_0, VISp_6]"
7,multiplane-ophys_782149_2025-04-08_09-28-39,2025-04-08,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[155, 195, 110, 235, 75, 275, 35, 300]",[VISp],fail,[VISp_6]


In [18]:
sessions.head()

,subject_id,session_id,name,session_type,stage,image_set,session_number,acquisition_date,session_date,session_time,age_days,genotype,sex,date_of_birth,rig,project_name,n_planes,plane_names,imaging_depths,targeted_structures,z_drift_qc_status,n_planes_failing_zdrift,planes_failing_zdrift,processed_stamp,_id
0,782149,multiplane-ophys_782149_2025-03-25_09-46-08,multiplane-ophys_782149_2025-03-25_09-46-08_pr...,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,2025-03-25,09:46:08.591468,108,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.2,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[40, 320, 80, 280, 120, 240, 160, 200]",[VISp],pass,0,[],2026-08-19_00-32-51,24865b94-ffd8-4eb9-b062-70fa46a17b39
1,782149,multiplane-ophys_782149_2025-03-28_10-55-25,multiplane-ophys_782149_2025-03-28_10-55-25_pr...,TRAINING_1_gratings,TRAINING_1,NaN,2,2025-03-28,2025-03-28,10:55:25.569080,111,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 114, 244, 80, 280, 40, 310]",[VISp],pass,0,[],2026-08-19_00-34-09,2bef1d11-dc35-4b8b-b825-7145bb0fe603
2,782149,multiplane-ophys_782149_2025-03-29_10-10-29,multiplane-ophys_782149_2025-03-29_10-10-29_pr...,TRAINING_1_gratings,TRAINING_1,NaN,3,2025-03-29,2025-03-29,10:10:29.493070,112,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[158, 198, 114, 246, 80, 276, 43, 306]",[VISp],fail,3,"[VISp_0, VISp_4, VISp_6]",2026-08-19_00-33-54,c1a6d250-dea5-4de5-8185-29a92e405878
3,782149,multiplane-ophys_782149_2025-03-31_12-23-33,multiplane-ophys_782149_2025-03-31_12-23-33_pr...,TRAINING_1_gratings,TRAINING_1,NaN,4,2025-03-31,2025-03-31,12:23:33.753970,114,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 245, 80, 280, 40, 310]",[VISp],pass,0,[],2026-08-19_00-34-28,edbe415f-da52-4f07-86b3-957fb399c4a6
4,782149,multiplane-ophys_782149_2025-04-01_09-42-11,multiplane-ophys_782149_2025-04-01_09-42-11_pr...,TRAINING_2_gratings_flashed,TRAINING_2,NaN,5,2025-04-01,2025-04-01,09:42:11.814685,115,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,MESO.1,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 240, 85, 280, 40, 300]",[VISp],fail,2,"[VISp_4, VISp_6]",2026-08-19_00-33-58,b670a19a-435f-401b-a63c-53ce8bd9c29f


## Write the CSV

Save to `/data/metadata/`: the data mounts are read-only inside a
capsule, and these tables are code-adjacent outputs that get versioned with the
notebooks.


In [19]:
session_csv = os.path.join(DATA_DIR, 'metadata', 'visual_learning_session_metadata.csv')
sessions.to_csv(session_csv, index=False)
print(f'{session_csv}  ({len(sessions)} rows, {sessions.shape[1]} columns)')

# plane_csv = os.path.join(DATA_DIR, 'metadata', 'visual_learning_plane_metadata.csv')
# planes.to_csv(plane_csv, index=False)
# print(f'{plane_csv}  ({len(planes)} rows, {planes.shape[1]} columns)')


/data/metadata/visual_learning_session_metadata.csv  (147 rows, 25 columns)


In [20]:
sessions.value_counts('subject_id')

subject_id
788406    32
800792    25
782149    24
790322    24
800995    22
804363    20
Name: count, dtype: int64

In [21]:
sessions.session_type.unique()

array(['TRAINING_0_gratings_autorewards_15min', 'TRAINING_1_gratings',
       'TRAINING_2_gratings_flashed', 'TRAINING_3_images_A_10uL_reward',
       'TRAINING_4_images_A_training', 'TRAINING_5_images_A_epilogue',
       'TRAINING_5_images_A_handoff_ready',
       'TRAINING_5_images_A_handoff_lapsed', 'OPHYS_1_images_A',
       'OPHYS_4_images_B', 'OPHYS_6_images_B', 'STAGE_0', 'STAGE_1'],
      dtype=object)

In [22]:
# check that test sessions are gone (should return none)
n_planes = pd.DataFrame(planes.groupby('name').count()['session_id'])
print(n_planes[n_planes.session_id!=8].index.unique())

Index([], dtype='object', name='name')


### Sanity checks before you trust these

docDB drops rows silently &mdash; it returns no error when an asset simply is not
indexed. Read the counts below against what you expect from the processing batch, and
if a mouse is short, re-run its aggregation rather than assuming the data is missing.


In [23]:
print('sessions per mouse')
print(sessions.subject_id.value_counts().sort_index().to_string())

print('\nplanes per session (should be 8 for most, 2 for late STAGE_1)')
print(planes.groupby('session_id').size().value_counts().to_string())

print('\nsession types')
print(sessions.session_type.value_counts().to_string())

missing_planes = set(sessions.session_id) - set(planes.session_id)
if missing_planes:
    print(f'\n{len(missing_planes)} sessions have no plane rows '
          f'(docDB indexing gap, not necessarily missing data)')


sessions per mouse
subject_id
782149    24
788406    32
790322    24
800792    25
800995    22
804363    20

planes per session (should be 8 for most, 2 for late STAGE_1)
8    147

session types
session_type
TRAINING_1_gratings                      26
TRAINING_3_images_A_10uL_reward          19
STAGE_1                                  19
OPHYS_6_images_B                         15
OPHYS_1_images_A                         12
OPHYS_4_images_B                         12
TRAINING_2_gratings_flashed               9
TRAINING_4_images_A_training              7
TRAINING_0_gratings_autorewards_15min     7
TRAINING_5_images_A_epilogue              7
STAGE_0                                   7
TRAINING_5_images_A_handoff_ready         6
TRAINING_5_images_A_handoff_lapsed        1
